Topic: Top 3 Products Per Category

Objective:
Find the top 3 products in each category based on total sales amount.


In [0]:
%sql
DROP TABLE IF EXISTS order_items;
DROP TABLE IF EXISTS products;

CREATE TABLE products (
    product_id INT PRIMARY KEY,
    product_name VARCHAR(100),
    category VARCHAR(50)
);

CREATE TABLE order_items (
    order_item_id INT PRIMARY KEY,
    order_id INT,
    product_id INT,
    quantity INT,
    unit_price DECIMAL(10, 2)
);

INSERT INTO products (product_id, product_name, category) VALUES
(1, 'Laptop', 'Electronics'),
(2, 'Mouse', 'Electronics'),
(3, 'Keyboard', 'Electronics'),
(4, 'Monitor', 'Electronics'),
(5, 'Office Chair', 'Furniture'),
(6, 'Desk', 'Furniture'),
(7, 'Desk Lamp', 'Furniture'),
(8, 'Bookshelf', 'Furniture'),
(9, 'Notebook', 'Stationery'),
(10, 'Pen', 'Stationery'),
(11, 'Marker', 'Stationery'),
(12, 'Stapler', 'Stationery');

INSERT INTO order_items (order_item_id, order_id, product_id, quantity, unit_price) VALUES
(101, 1001, 1, 2, 900.00),
(102, 1001, 2, 5, 25.00),
(103, 1002, 3, 3, 75.00),
(104, 1003, 4, 2, 300.00),
(105, 1004, 1, 1, 900.00),
(106, 1005, 5, 4, 150.00),
(107, 1006, 6, 2, 400.00),
(108, 1007, 7, 5, 45.00),
(109, 1008, 8, 1, 250.00),
(110, 1009, 9, 20, 3.00),
(111, 1010, 10, 50, 1.50),
(112, 1011, 11, 15, 2.00),
(113, 1012, 12, 5, 8.00),
(114, 1013, 2, 10, 25.00),
(115, 1014, 5, 2, 150.00),
(116, 1015, 6, 1, 400.00);



num_affected_rows,num_inserted_rows
16,16


In [0]:
%sql
SELECT 
    p.category,
    p.product_id,
    p.product_name,
    SUM(oi.quantity * oi.unit_price) AS total_sales
FROM products p 
INNER JOIN order_items oi
    ON p.product_id = oi.product_id
GROUP BY 
    p.category,
    p.product_id,
    p.product_name

category,product_id,product_name,total_sales
Stationery,10,Pen,75.00
Furniture,8,Bookshelf,250.00
Electronics,1,Laptop,2700.00
Stationery,9,Notebook,60.00
Electronics,2,Mouse,375.00
Furniture,7,Desk Lamp,225.00
Furniture,6,Desk,1200.00
Electronics,3,Keyboard,225.00
Electronics,4,Monitor,600.00
Stationery,12,Stapler,40.00


In [0]:
%sql
WITH product_sales AS (
    SELECT
        p.category,
        p.product_id,
        p.product_name,
        SUM(oi.quantity * oi.unit_price) AS total_sales
    FROM products p
    INNER JOIN order_items oi
        ON p.product_id = oi.product_id
    GROUP BY
        p.category,
        p.product_id,
        p.product_name
),

ranked_products AS (
    SELECT
        category,
        product_id,
        product_name,
        total_sales,
        DENSE_RANK() OVER (
            PARTITION BY category
            ORDER BY total_sales DESC
        ) AS sales_rank
    FROM product_sales
)

SELECT
    category,
    product_id,
    product_name,
    total_sales,
    sales_rank
FROM ranked_products
WHERE sales_rank <= 3
ORDER BY
    category,
    sales_rank,
    product_name;

category,product_id,product_name,total_sales,sales_rank
Electronics,1,Laptop,2700.00,1
Electronics,4,Monitor,600.00,2
Electronics,2,Mouse,375.00,3
Furniture,6,Desk,1200.00,1
Furniture,5,Office Chair,900.00,2
Furniture,8,Bookshelf,250.00,3
Stationery,10,Pen,75.00,1
Stationery,9,Notebook,60.00,2
Stationery,12,Stapler,40.00,3


In [0]:
%sql
-- ====================================================================
-- STEP 1: Aggregate raw sales data per product
-- This CTE(Common Table Expressions) calculates the total revenue generated by every individual product.
-- ====================================================================
WITH product_sales AS (
    SELECT 
        p.category, 
        p.product_id, 
        p.product_name,
        -- Calculate total sales revenue for each product (Quantity x Unit Price)
        SUM(oi.quantity * oi.unit_price) AS total_sales 
    FROM products p
    -- Join products with order items to match sales transactions with product details
    INNER JOIN order_items oi 
        ON p.product_id = oi.product_id
    -- Group by descriptive columns to isolate the sum for each unique product
    GROUP BY 
        p.category, 
        p.product_id, 
        p.product_name
), -- Comma acts as the bridge to define the next temporary table

-- ====================================================================
-- STEP 2: Rank the products within their respective categories
-- This CTE uses a window function to assign an ordered rank to each product.
-- ====================================================================
ranked_products AS (
    SELECT 
        category,
        product_id,
        product_name,
        total_sales,
        -- PARTITION BY groups the ranking context locally within each category.
        -- ORDER BY DESC ensures the highest sales amounts get a rank of 1.
        -- DENSE_RANK ensures if there's a tie, no rank numbers are skipped.
        DENSE_RANK() OVER (
            PARTITION BY category 
            ORDER BY total_sales DESC
        ) AS sales_rank
    FROM product_sales -- Pulls data directly from the first temporary table above
)

-- ====================================================================
-- STEP 3: Final Filter and Presentation
-- Pulls from the ranked table and strips away anything below the Top 3.
-- ====================================================================
SELECT 
    category,
    product_id,
    product_name,
    total_sales,
    sales_rank
FROM ranked_products
-- Filter out everything except the Top 3 products for each category
WHERE sales_rank <= 3
-- Sort the final output neatly by category, then by rank position
ORDER BY 
    category,
    sales_rank,
    product_name;

category,product_id,product_name,total_sales,sales_rank
Electronics,1,Laptop,2700.00,1
Electronics,4,Monitor,600.00,2
Electronics,2,Mouse,375.00,3
Furniture,6,Desk,1200.00,1
Furniture,5,Office Chair,900.00,2
Furniture,8,Bookshelf,250.00,3
Stationery,10,Pen,75.00,1
Stationery,9,Notebook,60.00,2
Stationery,12,Stapler,40.00,3
